In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Neural network example with LIME")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [2]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

# Преглед на данните
df.show(5)

+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|Gender|Age|   Degree|PlatformUsage|PlatformHours|OutPlatformHours|Satisfaction|Self-assessment|PreferredFormat|
+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|     Ж| 21|Бакалавър|           Да|            6|               2|           4|              4|          Видео|
|     М| 23| Магистър|           Да|            8|               1|           5|              5|          Видео|
|     Ж| 20|Бакалавър|           Не|            0|               3|           3|              3|          Текст|
|     Ж| 22|Бакалавър|           Да|            5|               1|           4|              4|         Смесен|
|     М| 24| Магистър|           Да|            9|               0|           5|              5|          Видео|
+------+---+---------+-------------+-------------+----------------+------------+---------------+

In [3]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Подготовка на данните
feature_cols = ["Age","PlatformHours","OutPlatformHours","Self-assessment"]
# Обединяване на всички характеристики във features колона
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df)

# Разделяне на данните
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

# Архитектура на мрежата
layers = [4, 7, 8, 6]
# Създаване на модела
mlp = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="Satisfaction",
    layers=layers, maxIter=100)
# Обучение
model = mlp.fit(train_data)

# Прогнозиране
predictions = model.transform(test_data)
# Оценка
evaluator = MulticlassClassificationEvaluator(
    labelCol="Satisfaction", predictionCol="prediction",
    metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

Accuracy: 0.25


In [11]:
from lime.lime_tabular import LimeTabularExplainer
import pandas as pd
import numpy as np

# Преобразуване към pandas
train_pd = train_data.select("Age","PlatformHours","OutPlatformHours","Self-Assessment").toPandas()
# Имена на признаците
feature_names = ["Age","PlatformHours","OutPlatformHours","Self-Assessment"]
# Класове
class_names = ["1", "2", "3", "4", "5"]

# Създаване на LIME обяснител
explainer = LimeTabularExplainer(
    training_data=train_pd.values, feature_names=feature_names,
    class_names=class_names,  mode='classification')
# Запис за обяснение
instance = train_pd.iloc[2].values

# Функция за вероятности
def predict_proba_fn(x):
    pdf = pd.DataFrame(x,columns=feature_names)

    spark_df = spark.createDataFrame(pdf)
    spark_df = assembler.transform(spark_df)
    predictions = model.transform(spark_df)

    return np.array(
        predictions.select("probability")
        .rdd
        .map(lambda row: row.probability.toArray())
        .collect()    )

# Генериране на обяснение
explanation = explainer.explain_instance(
    instance, predict_proba_fn, num_features=4)

# Извеждане на резултата
print(explanation.as_list())

[('3.75 < PlatformHours <= 6.50', -1.613334328889945e-214), ('1.00 < OutPlatformHours <= 2.00', 1.3599926984245211e-214), ('20.75 < Age <= 22.00', 5.559785398933436e-216), ('3.75 < Self-Assessment <= 4.00', 4.9114856005192405e-216)]


In [9]:
import sys
!{sys.executable} -m pip install lime

  Using cached lime-0.2.0.1.tar.gz (275 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached lazy_loader-0.5-py3-none-any.whl.metadata (5.9 kB)
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ----------------------- ---------------- 7.6/12.8 MB 39.3 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 32.1 MB/s  0:00:00
Using cached lazy_loader-0.5-py3-none-any.whl (8.0 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 31.1 MB/s  0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283917 sha256=7f432c6bdc74e719b14c2bd4920fab16691243975fbd8ac1e8a4e5

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
